In [1]:
# Install if not already
!pip install deap scikit-learn

import random
import numpy as np

from deap import base, creator, tools, algorithms
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# -------------------------------
# 1. DATASET (Replace with your dataset.csv)
# -------------------------------
# Example dummy dataset (for testing)
X = np.random.rand(200, 4)
y = X[:, 0]*2 + X[:, 1]*3 + np.random.rand(200)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# -------------------------------
# 2. GA SETUP
# -------------------------------
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Individual = [neurons, layers, learning_rate]
toolbox.register("neurons", random.randint, 5, 100)
toolbox.register("layers", random.randint, 1, 3)
toolbox.register("lr", random.uniform, 0.0001, 0.1)

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.neurons, toolbox.layers, toolbox.lr), n=1)

toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# -------------------------------
# 3. FITNESS FUNCTION (IMPORTANT)
# -------------------------------
def evaluate(individual):
    neurons, layers, lr = individual
    
    # Build NN architecture
    hidden_layers = tuple([neurons] * layers)
    
    try:
        model = MLPRegressor(hidden_layer_sizes=hidden_layers,
                             learning_rate_init=lr,
                             max_iter=300,
                             random_state=0)
        
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        error = mean_squared_error(y_test, preds)
        
    except:
        # If model fails (rare case), assign high error
        error = 1e6
    
    return (error,)

toolbox.register("evaluate", evaluate)

# Genetic operators
toolbox.register("mate", tools.cxTwoPoint)

toolbox.register("mutate", tools.mutUniformInt,
                 low=[5, 1, 0],
                 up=[100, 3, 1],
                 indpb=0.2)

toolbox.register("select", tools.selTournament, tournsize=3)

# -------------------------------
# 4. RUN GA
# -------------------------------
POP_SIZE = 10
GENS = 5

population = toolbox.population(n=POP_SIZE)

for gen in range(GENS):
    print(f"\nGeneration {gen+1}")
    
    offspring = algorithms.varAnd(population, toolbox, cxpb=0.6, mutpb=0.3)
    
    fitnesses = list(map(toolbox.evaluate, offspring))
    
    for ind, fit in zip(offspring, fitnesses):
        ind.fitness.values = fit
    
    population = toolbox.select(offspring, k=len(population))

# -------------------------------
# 5. BEST RESULT
# -------------------------------
best = tools.selBest(population, k=1)[0]

print("\nBest Parameters Found:")
print("Neurons:", best[0])
print("Layers:", best[1])
print("Learning Rate:", best[2])


Generation 1

Generation 2

Generation 3

Generation 4

Generation 5

Best Parameters Found:
Neurons: 48
Layers: 2
Learning Rate: 0.03329206470140072
